## Imports

In [1]:
!mkdir -p /root/.cache/torch/hub/checkpoints
!cp -r ../input/landmark-additional-packages/rwightman_gen-efficientnet-pytorch_master/rwightman_gen-efficientnet-pytorch_master /root/.cache/torch/hub
!cp ../input/landmark-additional-packages/tf_efficientnet_b3_aa-84b4657e.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/landmark-additional-packages/tf_efficientnet_b5_ra-9a3e5369.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/landmark-additional-packages/se_resnext50_32x4d-a260b3a4.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/landmark-additional-packages/resnet50d_ra2-464e36ba.pth /root/.cache/torch/hub/checkpoints/

In [2]:
!pip install -q ../input/landmark-additional-packages/timm-0.3.4-py3-none-any.whl
!pip install -q ../input/landmark-additional-packages/geffnet-1.0.0-py3-none-any.whl
!pip install -q ../input/landmark-additional-packages/EfficientNet-PyTorch/EfficientNet-PyTorch-master
!pip install -q ../input/landmark-additional-packages/pycocotools-2.0.2/dist/pycocotools-2.0.2.tar
!pip install -q ../input/landmark-additional-packages/pretrainedmodels-0.7.4/pretrainedmodels-0.7.4

In [3]:
!pip install "/kaggle/input/hpamisc/pytorch_zoo-master"
!pip install "/kaggle/input/hpamisc/pycocotools-2.0-cp37-cp37m-linux_x86_64.whl"
!pip install "/kaggle/input/hpamisc/faiss_gpu-1.7.0-cp37-cp37m-manylinux2014_x86_64.whl"

ERROR: Invalid requirement: '/kaggle/input/hpamisc/pytorch_zoo-master'
Hint: It looks like a path. File '/kaggle/input/hpamisc/pytorch_zoo-master' does not exist.
Processing /kaggle/input/hpamisc/pycocotools-2.0-cp37-cp37m-linux_x86_64.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/hpamisc/pycocotools-2.0-cp37-cp37m-linux_x86_64.whl'

Processing /kaggle/input/hpamisc/faiss_gpu-1.7.0-cp37-cp37m-manylinux2014_x86_64.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/hpamisc/faiss_gpu-1.7.0-cp37-cp37m-manylinux2014_x86_64.whl'



In [4]:
! python ../input/maozi-no-arcface/maozi_no_arcface.py

Traceback (most recent call last):
  File "../input/maozi-no-arcface/maozi_no_arcface.py", line 10, in <module>
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 724, in exec_module
  File "<frozen importlib._bootstrap_external>", line 859, in get_code
  File "<frozen importlib._bootstrap_external>", line 916, in get_data
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/timmilya/pytorch-image-models-master/timm/__init__.py'


In [5]:
import random
import sys
sys.path.append('../input/hpa-singlecell-e050f56/hpa_singlecell-double_level_valid_all/')

from torch import nn
import torch
import torch.nn.functional as F
import torchvision
import timm
from torch.nn.parameter import Parameter
import albumentations as A

from utils import parse_args, prepare_for_result
from torch.utils.data import DataLoader, Dataset
from losses import get_loss, get_class_balanced_weighted
from dataloaders import get_dataloader
from utils import load_matched_state
from configs import Config
from models import get_model
from dataloaders.transform_loader import get_tfms

import base64
import zlib
from pycocotools import _mask as coco_mask
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cv2
import tqdm
import seaborn as sns

In [6]:
import torchvision
print(torchvision.__version__)

from torchvision import transforms

0.8.1


## Transforms

In [7]:
tensor_tfms = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406, 0.406], 
                                             std=[0.229, 0.224, 0.225, 0.225]),
        ])

image_tfms = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406, 0.406], 
                                             std=[0.229, 0.224, 0.225, 0.225]),
        ])

tta_tfms = A.Compose([
    A.Resize(always_apply=False, p=1, height=256, width=256, interpolation=1),
    A.HorizontalFlip(always_apply=False, p=0.5),
    A.ShiftScaleRotate(always_apply=False, p=0.7, shift_limit_x=(-0.06, 0.06), shift_limit_y=(-0.06, 0.06), scale_limit=(-0.3, 0.3), rotate_limit=(-22.5, 22.5), interpolation=1, border_mode=2, value=None, mask_value=None),
    A.RandomBrightnessContrast(always_apply=False, p=0.5, brightness_limit=(-0.2, 0.2), contrast_limit=(-0.2, 0.2), brightness_by_max=True),
])

res_tfms = A.Compose([
    A.Resize(always_apply=False, p=1, height=256, width=256, interpolation=1)
])

image_res_tfms = A.Compose([
    A.Resize(always_apply=False, p=1, height=512, width=512, interpolation=1)
])

## Seed

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using GPU

    # For convolutional determinism
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

## Functions

In [9]:
def binary_mask_to_ascii(mask, mask_val=1):
    """Converts a binary mask into OID challenge encoding ascii text."""
    mask = np.where(mask==mask_val, 1, 0).astype(np.bool)
    
    # check input mask --
    if mask.dtype != np.bool:
        raise ValueError(f"encode_binary_mask expects a binary mask, received dtype == {mask.dtype}")

    mask = np.squeeze(mask)
    if len(mask.shape) != 2:
        raise ValueError(f"encode_binary_mask expects a 2d mask, received shape == {mask.shape}")

    # convert input mask to expected COCO API input --
    mask_to_encode = mask.reshape(mask.shape[0], mask.shape[1], 1)
    mask_to_encode = mask_to_encode.astype(np.uint8)
    mask_to_encode = np.asfortranarray(mask_to_encode)

    # RLE encode mask --
    encoded_mask = coco_mask.encode(mask_to_encode)[0]["counts"]

    # compress and base64 encoding --
    binary_str = zlib.compress(encoded_mask, zlib.Z_BEST_COMPRESSION)
    base64_str = base64.b64encode(binary_str)
    return base64_str.decode()

def process(x):
    iid, msk, img, sz = x
    img = cv2.resize(img, (2048, 2048))
    enc_msk = cv2.resize(msk, (sz, sz))
    cell_mask = msk
    subs = {}
    results = []
    for i in range(1, cell_mask.max() + 1):
        enc = binary_mask_to_ascii(enc_msk, i)
        sub = cv2.resize((cell_mask == i).astype(np.float), (2048, 2048), cv2.INTER_LINEAR)
        xr, yr = np.where(sub == 1)
        xmin, xmax, ymin, ymax = xr.min(), xr.max(), yr.min(), yr.max()
        subs[i] = (img * np.repeat((sub == 1).astype(np.int)[:, :, np.newaxis], 4, 2))[xmin:xmax, ymin: ymax]
#         imsave(f'./seg_png_fix_test/{iid}_{i}.png', (255 * subs[i]).astype(np.uint8))
        results.append(((255 * subs[i]).astype(np.uint8), enc, sz, sz))
    return results

def squarify(M,val):
    (a,b,c)=M.shape
    if a>b:
        padding=((0,0),((a-b)//2,a-b-(a-b)//2),(0, 0))
    else:
        padding=(((b-a)//2,b-a-(b-a)//2),(0,0),(0, 0))
    return np.pad(M,padding,mode='constant',constant_values=val)

In [10]:
CLASSES = np.asarray([
'0. Nucleoplasm',
'1. Nuclear-membrane',
'2. Nucleoli',
'3. Nucleoli-fibrillar-center',
'4. Nuclear-speckles',
'5. Nuclear-bodies',
'6. Endoplasmic-reticulum',
'7. Golgi apparatus',
'8. Intermediate-filaments',
'9. Actin-filaments',
'10. Microtubules',
'11. Mitotic-spindle',
'12. Centrosome',
'13. Plasma-membrane',
'14. Mitochondria',
'15. Aggresome',
'16. Cytosol',
'17. Vesicles',
'18. Negative'
])


def get_image(images_dir, sample_id):
    colors = ('red','green','blue','yellow')
    images = [cv2.imread(os.path.join(images_dir, f'{sample_id}_{c}.png'), cv2.IMREAD_GRAYSCALE) for c in colors]
    image = np.stack(images, axis=-1)
    if image.dtype == np.uint16:
        image = ((image.astype("float32")/65536.)*255.).astype("uint8")
    return image

def get_masks(image_ids, imgs=None, cell_df=None):
    if not isinstance(image_ids, (tuple, list, set)): image_ids = [image_ids]
    if not isinstance(imgs, (tuple, list, set)): imgs = [imgs]
        
    all_samples_precomputed = all(_id in cell_df.index for _id in image_ids)
    if all_samples_precomputed:
        masks = []
        for image_id in image_ids:
            (W, H, pred_str) = cell_df.loc[image_id]
            parts = pred_str.split()
            rles = parts[2::3]
            mask = coco_mask.decode([{"size": [H, W], "counts": zlib.decompress(base64.b64decode(r))} for r in rles])
            mask = mask.argmax(-1)
            masks.append(mask)
        return masks
    # return [None]*len(imgs)

    images = [[img[:, :, 0] for img in imgs],
              [img[:, :, 3] for img in imgs], 
              [img[:, :, 2] for img in imgs]]

    nuc_segmentations = segmentator.pred_nuclei(images[2])
    cell_segmentations = segmentator.pred_cells(images)
    cell_masks = []
    for i in range(len(cell_segmentations)):
        _, cell_mask = label_cell(nuc_segmentations[i], cell_segmentations[i])
        cell_masks.append(cell_mask)
    return cell_masks

## ResNest269 Model

In [11]:
#@title Model / Architecture

import math
import random

import cv2
import numpy as np

import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import BatchNorm2d, Conv2d, Linear, Module, ReLU
from torch.nn.modules.utils import _pair
import torch.utils.model_zoo as model_zoo
from torch.optim.lr_scheduler import LambdaLR


def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)

  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


def rotation(x, k):
  return torch.rot90(x, k, (1, 2))


def interleave(x, size):
  s = list(x.shape)
  return x.reshape([-1, size] + s[1:]).transpose(0, 1).reshape([-1] + s[1:])


def de_interleave(x, size):
  s = list(x.shape)
  return x.reshape([size, -1] + s[1:]).transpose(0, 1).reshape([-1] + s[1:])


def resize_tensor(tensors, size, mode='bilinear', align_corners=None):
  return F.interpolate(tensors, size, mode=mode, align_corners=align_corners)


def gap2d(x, keepdims=False):
  x = torch.mean(x.view(x.size(0), x.size(1), -1), -1)
  if keepdims:
    x = x.view(x.size(0), x.size(1), 1, 1)
  return x


# Losses


def L1_Loss(A_tensors, B_tensors):
  return torch.abs(A_tensors - B_tensors)


def L2_Loss(A_tensors, B_tensors):
  return torch.pow(A_tensors - B_tensors, 2)


# ratio = 0.2, top=20%
def Online_Hard_Example_Mining(values, ratio=0.2):
  b, c, h, w = values.size()
  return torch.topk(values.reshape(b, -1), k=int(c * h * w * ratio), dim=-1)[0]


def shannon_entropy_loss(logits, activation=torch.sigmoid, epsilon=1e-5):
  v = activation(logits)
  return -torch.sum(v * torch.log(v + epsilon), dim=1).mean()


def make_cam(x, eps=1e-5, shift_min=False, global_norm=False, inplace=True):
  x = F.relu(x)

  if global_norm:
    x_min = x.min() if shift_min else 0
    x_max = x.max() - x_min
  else:
    b, c, h, w = x.size()
    flat_x = x.view(b, c, -1)
    x_min = flat_x.min(axis=-1)[0].view((b, c, 1, 1)) if shift_min else 0
    x_max = flat_x.max(axis=-1)[0].view((b, c, 1, 1)) - x_min

  if shift_min:
    if inplace:
      x -= x_min
      x /= x_max + eps
    else:
      x = (x - x_min) / (x_max + eps)
  else:
    if inplace:
      x /= x_max + eps
    else:
      x = x / (x_max + eps)

  return x


def one_hot_embedding(label, classes):
  """Embedding labels to one-hot form.

    Args:
      labels: (int) class labels.
      num_classes: (int) number of classes.

    Returns:
      (tensor) encoded labels, sized [N, #classes].
    """

  vector = np.zeros((classes), dtype=np.float32)
  if len(label) > 0:
    vector[label] = 1.
  return vector


def calculate_parameters(model):
  return sum(param.numel() for param in model.parameters()) / 1000000.0


def get_learning_rate_from_optimizer(optimizer):
  return optimizer.param_groups[0]['lr']


def set_trainable_layers(model, klass=None, trainable=False):
  for m in model.modules():
    if klass is None or isinstance(m, klass):
      for prop in ("weight", "bias"):
        w = getattr(m, prop, None)
        if w is not None:
          w.requires_grad = trainable


def to_numpy(tensor):
  return tensor.cpu().detach().numpy()


def load_model(model, model_path, parallel=False, map_location=None, strict=True):
  print(f'loading weights from `{model_path}`.')

  state = torch.load(model_path, map_location=map_location)
  if parallel:
    model.module.load_state_dict(state, strict=strict)
  else:
    model.load_state_dict(state, strict=strict)


def save_model(model, model_path, parallel=False):
  print(f'saving weights to `{model_path}`.')

  if parallel:
    torch.save(model.module.state_dict(), model_path)
  else:
    torch.save(model.state_dict(), model_path)


def transfer_model(pretrained_model, model):
  pretrained_dict = pretrained_model.state_dict()
  model_dict = model.state_dict()

  pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict}

  model_dict.update(pretrained_dict)
  model.load_state_dict(model_dict)


def get_learning_rate(optimizer):
  lr = []
  for param_group in optimizer.param_groups:
    lr += [param_group['lr']]
  return lr


def get_cosine_schedule_with_warmup(optimizer, warmup_iteration, max_iteration, cycles=7. / 16.):

  def _lr_lambda(current_iteration):
    if current_iteration < warmup_iteration:
      return float(current_iteration) / float(max(1, warmup_iteration))

    no_progress = float(current_iteration - warmup_iteration) / float(max(1, max_iteration - warmup_iteration))
    return max(0., math.cos(math.pi * cycles * no_progress))

  return LambdaLR(optimizer, _lr_lambda, -1)


def label_smoothing(labels, alpha):
  if alpha:
    return (1 - alpha) * labels + alpha * 0.5

  return labels


class SplAtConv2d(Module):
  """Split-Attention Conv2d
    """

  def __init__(
    self,
    in_channels,
    channels,
    kernel_size,
    stride=(1, 1),
    padding=(0, 0),
    dilation=(1, 1),
    groups=1,
    bias=True,
    radix=2,
    reduction_factor=4,
    rectify=False,
    rectify_avg=False,
    norm_layer=None,
    dropblock_prob=0.0,
    **kwargs
  ):
    super(SplAtConv2d, self).__init__()
    padding = _pair(padding)
    self.rectify = rectify and (padding[0] > 0 or padding[1] > 0)
    self.rectify_avg = rectify_avg
    inter_channels = max(in_channels * radix // reduction_factor, 32)
    self.radix = radix
    self.cardinality = groups
    self.channels = channels
    self.dropblock_prob = dropblock_prob
    if self.rectify:
      from rfconv import RFConv2d
      self.conv = RFConv2d(
        in_channels,
        channels * radix,
        kernel_size,
        stride,
        padding,
        dilation,
        groups=groups * radix,
        bias=bias,
        average_mode=rectify_avg,
        **kwargs
      )
    else:
      self.conv = Conv2d(
        in_channels,
        channels * radix,
        kernel_size,
        stride,
        padding,
        dilation,
        groups=groups * radix,
        bias=bias,
        **kwargs
      )
    self.use_bn = norm_layer is not None
    if self.use_bn:
      self.bn0 = norm_layer(channels * radix)
    self.relu = ReLU(inplace=True)
    self.fc1 = Conv2d(channels, inter_channels, 1, groups=self.cardinality)
    if self.use_bn:
      self.bn1 = norm_layer(inter_channels)
    self.fc2 = Conv2d(inter_channels, channels * radix, 1, groups=self.cardinality)
    if dropblock_prob > 0.0:
      self.dropblock = DropBlock2D(dropblock_prob, 3)
    self.rsoftmax = rSoftMax(radix, groups)

  def forward(self, x):
    x = self.conv(x)
    if self.use_bn:
      x = self.bn0(x)
    if self.dropblock_prob > 0.0:
      x = self.dropblock(x)
    x = self.relu(x)

    batch, rchannel = x.shape[:2]
    if self.radix > 1:
      if torch.__version__ < '1.5':
        splited = torch.split(x, int(rchannel // self.radix), dim=1)
      else:
        splited = torch.split(x, rchannel // self.radix, dim=1)
      gap = sum(splited)
    else:
      gap = x
    gap = F.adaptive_avg_pool2d(gap, 1)
    gap = self.fc1(gap)

    if self.use_bn:
      gap = self.bn1(gap)
    gap = self.relu(gap)

    atten = self.fc2(gap)
    atten = self.rsoftmax(atten).view(batch, -1, 1, 1)

    if self.radix > 1:
      if torch.__version__ < '1.5':
        attens = torch.split(atten, int(rchannel // self.radix), dim=1)
      else:
        attens = torch.split(atten, rchannel // self.radix, dim=1)
      out = sum([att * split for (att, split) in zip(attens, splited)])
    else:
      out = atten * x
    return out.contiguous()


class rSoftMax(nn.Module):

  def __init__(self, radix, cardinality):
    super().__init__()
    self.radix = radix
    self.cardinality = cardinality

  def forward(self, x):
    batch = x.size(0)
    if self.radix > 1:
      x = x.view(batch, self.cardinality, self.radix, -1).transpose(1, 2)
      x = F.softmax(x, dim=1)
      x = x.reshape(batch, -1)
    else:
      x = torch.sigmoid(x)
    return x


class DropBlock2D(object):

  def __init__(self, *args, **kwargs):
    raise NotImplementedError


class GlobalAvgPool2d(nn.Module):

  def __init__(self):
    """Global average pooling over the input's spatial dimensions"""
    super(GlobalAvgPool2d, self).__init__()

  def forward(self, inputs):
    return nn.functional.adaptive_avg_pool2d(inputs, 1).view(inputs.size(0), -1)


class Bottleneck(nn.Module):
  """ResNet Bottleneck
    """
  # pylint: disable=unused-argument
  expansion = 4

  def __init__(
    self,
    inplanes,
    planes,
    stride=1,
    downsample=None,
    radix=1,
    cardinality=1,
    bottleneck_width=64,
    avd=False,
    avd_first=False,
    dilation=1,
    is_first=False,
    rectified_conv=False,
    rectify_avg=False,
    norm_layer=None,
    dropblock_prob=0.0,
    last_gamma=False
  ):
    super(Bottleneck, self).__init__()
    group_width = int(planes * (bottleneck_width / 64.)) * cardinality
    self.conv1 = nn.Conv2d(inplanes, group_width, kernel_size=1, bias=False)
    self.bn1 = norm_layer(group_width)
    self.dropblock_prob = dropblock_prob
    self.radix = radix
    self.avd = avd and (stride > 1 or is_first)
    self.avd_first = avd_first

    if self.avd:
      self.avd_layer = nn.AvgPool2d(3, stride, padding=1)
      stride = 1

    if dropblock_prob > 0.0:
      self.dropblock1 = DropBlock2D(dropblock_prob, 3)
      if radix == 1:
        self.dropblock2 = DropBlock2D(dropblock_prob, 3)
      self.dropblock3 = DropBlock2D(dropblock_prob, 3)

    if radix >= 1:
      self.conv2 = SplAtConv2d(
        group_width,
        group_width,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        dilation=dilation,
        groups=cardinality,
        bias=False,
        radix=radix,
        rectify=rectified_conv,
        rectify_avg=rectify_avg,
        norm_layer=norm_layer,
        dropblock_prob=dropblock_prob
      )
    elif rectified_conv:
      from rfconv import RFConv2d
      self.conv2 = RFConv2d(
        group_width,
        group_width,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        dilation=dilation,
        groups=cardinality,
        bias=False,
        average_mode=rectify_avg
      )
      self.bn2 = norm_layer(group_width)
    else:
      self.conv2 = nn.Conv2d(
        group_width,
        group_width,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        dilation=dilation,
        groups=cardinality,
        bias=False
      )
      self.bn2 = norm_layer(group_width)

    self.conv3 = nn.Conv2d(group_width, planes * 4, kernel_size=1, bias=False)
    self.bn3 = norm_layer(planes * 4)

    if last_gamma:
      from torch.nn.init import zeros_
      zeros_(self.bn3.weight)
    self.relu = nn.ReLU(inplace=True)
    self.downsample = downsample
    self.dilation = dilation
    self.stride = stride

  def forward(self, x):
    residual = x

    out = self.conv1(x)
    out = self.bn1(out)
    if self.dropblock_prob > 0.0:
      out = self.dropblock1(out)
    out = self.relu(out)

    if self.avd and self.avd_first:
      out = self.avd_layer(out)

    out = self.conv2(out)
    if self.radix == 0:
      out = self.bn2(out)
      if self.dropblock_prob > 0.0:
        out = self.dropblock2(out)
      out = self.relu(out)

    if self.avd and not self.avd_first:
      out = self.avd_layer(out)

    out = self.conv3(out)
    out = self.bn3(out)
    if self.dropblock_prob > 0.0:
      out = self.dropblock3(out)

    if self.downsample is not None:
      residual = self.downsample(x)

    out += residual
    out = self.relu(out)

    return out


class ResNet(nn.Module):
  """ResNet Variants

    Parameters
    ----------
    block : Block
        Class for the residual block. Options are BasicBlockV1, BottleneckV1.
    layers : list of int
        Numbers of layers in each block
    classes : int, default 1000
        Number of classification classes.
    dilated : bool, default False
        Applying dilation strategy to pretrained ResNet yielding a stride-8 model,
        typically used in Semantic Segmentation.
    norm_layer : object
        Normalization layer used in backbone network (default: :class:`mxnet.gluon.nn.BatchNorm`;
        for Synchronized Cross-GPU BachNormalization).

    Reference:

        - He, Kaiming, et al. "Deep residual learning for image recognition." Proceedings of the IEEE conference on computer vision and pattern recognition. 2016.

        - Yu, Fisher, and Vladlen Koltun. "Multi-scale context aggregation by dilated convolutions."
    """

  # pylint: disable=unused-variable
  def __init__(
    self,
    block,
    layers,
    radix=1,
    groups=1,
    bottleneck_width=64,
    num_classes=1000,
    dilated=False,
    dilation=1,
    deep_stem=False,
    stem_width=64,
    avg_down=False,
    rectified_conv=False,
    rectify_avg=False,
    avd=False,
    avd_first=False,
    final_drop=0.0,
    dropblock_prob=0,
    last_gamma=False,
    norm_layer=nn.BatchNorm2d
  ):
    self.cardinality = groups
    self.bottleneck_width = bottleneck_width
    # ResNet-D params
    self.stage_features = []
    self.outplanes = self.inplanes = stem_width * 2 if deep_stem else 64
    self.avg_down = avg_down
    self.last_gamma = last_gamma
    # ResNeSt params
    self.radix = radix
    self.avd = avd
    self.avd_first = avd_first

    super(ResNet, self).__init__()
    self.rectified_conv = rectified_conv
    self.rectify_avg = rectify_avg
    if rectified_conv:
      from rfconv import RFConv2d
      conv_layer = RFConv2d
    else:
      conv_layer = nn.Conv2d
    conv_kwargs = {'average_mode': rectify_avg} if rectified_conv else {}
    if deep_stem:
      self.conv1 = nn.Sequential(
        conv_layer(3, stem_width, kernel_size=3, stride=2, padding=1, bias=False, **conv_kwargs),
        norm_layer(stem_width),
        nn.ReLU(inplace=True),
        conv_layer(stem_width, stem_width, kernel_size=3, stride=1, padding=1, bias=False, **conv_kwargs),
        norm_layer(stem_width),
        nn.ReLU(inplace=True),
        conv_layer(stem_width, stem_width * 2, kernel_size=3, stride=1, padding=1, bias=False, **conv_kwargs),
      )
    else:
      self.conv1 = conv_layer(3, 64, kernel_size=7, stride=2, padding=3, bias=False, **conv_kwargs)
    self.bn1 = norm_layer(self.inplanes)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    self.layer1 = self._make_layer(block, 64, layers[0], norm_layer=norm_layer, is_first=False)
    self.layer2 = self._make_layer(block, 128, layers[1], stride=2, norm_layer=norm_layer)
    if dilated or dilation == 4:
      self.layer3 = self._make_layer(
        block, 256, layers[2], stride=1, dilation=2, norm_layer=norm_layer, dropblock_prob=dropblock_prob
      )
      self.layer4 = self._make_layer(
        block, 512, layers[3], stride=1, dilation=4, norm_layer=norm_layer, dropblock_prob=dropblock_prob
      )
    elif dilation == 2:
      self.layer3 = self._make_layer(
        block, 256, layers[2], stride=2, dilation=1, norm_layer=norm_layer, dropblock_prob=dropblock_prob
      )
      self.layer4 = self._make_layer(
        block, 512, layers[3], stride=1, dilation=2, norm_layer=norm_layer, dropblock_prob=dropblock_prob
      )
    else:
      self.layer3 = self._make_layer(
        block, 256, layers[2], stride=2, norm_layer=norm_layer, dropblock_prob=dropblock_prob
      )
      self.layer4 = self._make_layer(
        block, 512, layers[3], stride=2, norm_layer=norm_layer, dropblock_prob=dropblock_prob
      )

    self.avgpool = GlobalAvgPool2d()
    self.drop = nn.Dropout(final_drop) if final_drop > 0.0 else None
    self.fc = nn.Linear(512 * block.expansion, num_classes)

    for m in self.modules():
      if isinstance(m, nn.Conv2d):
        n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
        m.weight.data.normal_(0, math.sqrt(2. / n))
      elif isinstance(m, norm_layer):
        m.weight.data.fill_(1)
        m.bias.data.zero_()

  def _make_layer(
    self, block, planes, blocks, stride=1, dilation=1, norm_layer=None, dropblock_prob=0.0, is_first=True
  ):
    downsample = None
    if stride != 1 or self.inplanes != planes * block.expansion:
      down_layers = []
      if self.avg_down:
        if dilation == 1:
          down_layers.append(nn.AvgPool2d(kernel_size=stride, stride=stride, ceil_mode=True, count_include_pad=False))
        else:
          down_layers.append(nn.AvgPool2d(kernel_size=1, stride=1, ceil_mode=True, count_include_pad=False))
        down_layers.append(nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=1, bias=False))
      else:
        down_layers.append(nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False))
      down_layers.append(norm_layer(planes * block.expansion))
      downsample = nn.Sequential(*down_layers)

    layers = []
    if dilation == 1 or dilation == 2:
      layers.append(
        block(
          self.inplanes,
          planes,
          stride,
          downsample=downsample,
          radix=self.radix,
          cardinality=self.cardinality,
          bottleneck_width=self.bottleneck_width,
          avd=self.avd,
          avd_first=self.avd_first,
          dilation=1,
          is_first=is_first,
          rectified_conv=self.rectified_conv,
          rectify_avg=self.rectify_avg,
          norm_layer=norm_layer,
          dropblock_prob=dropblock_prob,
          last_gamma=self.last_gamma
        )
      )
    elif dilation == 4:
      layers.append(
        block(
          self.inplanes,
          planes,
          stride,
          downsample=downsample,
          radix=self.radix,
          cardinality=self.cardinality,
          bottleneck_width=self.bottleneck_width,
          avd=self.avd,
          avd_first=self.avd_first,
          dilation=2,
          is_first=is_first,
          rectified_conv=self.rectified_conv,
          rectify_avg=self.rectify_avg,
          norm_layer=norm_layer,
          dropblock_prob=dropblock_prob,
          last_gamma=self.last_gamma
        )
      )
    else:
      raise RuntimeError("=> unknown dilation size: {}".format(dilation))

    self.outplanes = self.inplanes = planes * block.expansion
    self.stage_features.append(self.outplanes)

    for i in range(1, blocks):
      layers.append(
        block(
          self.inplanes,
          planes,
          radix=self.radix,
          cardinality=self.cardinality,
          bottleneck_width=self.bottleneck_width,
          avd=self.avd,
          avd_first=self.avd_first,
          dilation=dilation,
          rectified_conv=self.rectified_conv,
          rectify_avg=self.rectify_avg,
          norm_layer=norm_layer,
          dropblock_prob=dropblock_prob,
          last_gamma=self.last_gamma
        )
      )

    return nn.Sequential(*layers)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.maxpool(x)

    x1 = self.layer1(x)
    x2 = self.layer2(x1)
    x3 = self.layer3(x2)
    x4 = self.layer4(x3)

    # print(x.size())

    # x = self.avgpool(x)
    #x = x.view(x.size(0), -1)
    # x = torch.flatten(x, 1)
    # if self.drop:
    #   x = self.drop(x)
    # x = self.fc(x)

    return (x1, x2, x3, x4)

_url_format = 'https://github.com/zhanghang1989/ResNeSt/releases/download/weights_step1/{}-{}.pth'

_model_sha256 = {
  name: checksum for checksum, name in [
    ('528c19ca', 'resnest50'),
    ('22405ba7', 'resnest101'),
    ('75117900', 'resnest200'),
    ('0cc87c48', 'resnest269'),
  ]
}


def short_hash(name):
  if name not in _model_sha256:
    raise ValueError('Pretrained model for {name} is not available.'.format(name=name))
  return _model_sha256[name][:8]


resnest_model_urls = {name: _url_format.format(name, short_hash(name)) for name in _model_sha256.keys()}


def resnest50(pretrained=False, root='~/.encoding/models', **kwargs):
  model = ResNet(
    Bottleneck, [3, 4, 6, 3],
    radix=2,
    groups=1,
    bottleneck_width=64,
    deep_stem=True,
    stem_width=32,
    avg_down=True,
    avd=True,
    avd_first=False,
    **kwargs
  )
  if pretrained:
    model.load_state_dict(
      torch.hub.load_state_dict_from_url(resnest_model_urls['resnest50'], progress=True, check_hash=True)
    )
  return model


def resnest101(pretrained=False, root='~/.encoding/models', **kwargs):
  model = ResNet(
    Bottleneck, [3, 4, 23, 3],
    radix=2,
    groups=1,
    bottleneck_width=64,
    deep_stem=True,
    stem_width=64,
    avg_down=True,
    avd=True,
    avd_first=False,
    **kwargs
  )
  if pretrained:
    model.load_state_dict(
      torch.hub.load_state_dict_from_url(resnest_model_urls['resnest101'], progress=True, check_hash=True)
    )
  return model


def resnest200(pretrained=False, root='~/.encoding/models', **kwargs):
  model = ResNet(
    Bottleneck, [3, 24, 36, 3],
    radix=2,
    groups=1,
    bottleneck_width=64,
    deep_stem=True,
    stem_width=64,
    avg_down=True,
    avd=True,
    avd_first=False,
    **kwargs
  )
  if pretrained:
    model.load_state_dict(
      torch.hub.load_state_dict_from_url(resnest_model_urls['resnest200'], progress=True, check_hash=True)
    )
  return model


def resnest269(pretrained=False, root='~/.encoding/models', **kwargs):
  model = ResNet(
    Bottleneck, [3, 30, 48, 8],
    radix=2,
    groups=1,
    bottleneck_width=64,
    deep_stem=True,
    stem_width=64,
    avg_down=True,
    avd=True,
    avd_first=False,
    **kwargs
  )
  if pretrained:
    model.load_state_dict(
      torch.hub.load_state_dict_from_url(resnest_model_urls['resnest269'], progress=True, check_hash=True)
    )
  return model


class FixedBatchNorm(nn.BatchNorm2d):

  def forward(self, x):
    return F.batch_norm(x, self.running_mean, self.running_var, self.weight, self.bias, training=False, eps=self.eps)


def group_norm(features):
  return nn.GroupNorm(4, features)


#######################################################################

def patch_conv_in_channels(model, layer_name, new_in_channels, copying_channel=0):
  layer = getattr(model, layer_name)  # layer = model.conv1
  # new_layer = layer.clone().detach()

  if isinstance(layer, nn.Sequential):
    cv_layers = list(layer.children())
    cv = cv_layers[0]
  else:
    cv = layer

  if not isinstance(cv, nn.Conv2d):
    raise ValueError(f"Cannot extract Conv2d from {cv}.")

  new_cv = nn.Conv2d(
    in_channels=new_in_channels,
    out_channels=cv.out_channels,
    kernel_size=cv.kernel_size,
    stride=cv.stride,
    padding=cv.padding,
    bias=cv.bias).requires_grad_()

  with torch.no_grad():
    new_cv.weight[:, :cv.in_channels, :, :] = cv.weight.data

    for i in range(new_in_channels - cv.in_channels):
        channel = cv.in_channels + i
        new_cv.weight[:, channel:channel+1, :, :] = cv.weight[:, copying_channel:copying_channel+1, : :].data
  new_cv.weight = nn.Parameter(new_cv.weight)

  if isinstance(layer, nn.Sequential):
    new_cv = nn.Sequential(new_cv, *cv_layers[1:])

  setattr(model, layer_name, new_cv)  # model.conv1 = new_layer


def build_backbone(name, dilated, strides, norm_fn, weights='imagenet', channels=3, **kwargs):
  dilation = 4 if dilated else 2

  pretrained = weights == "imagenet"
  model_fn = globals()[name]
  model = model_fn(pretrained=pretrained, dilated=dilated, dilation=dilation, norm_layer=norm_fn)
  if channels != 3:
    patch_conv_in_channels(model, "conv1", channels)
  if pretrained:
    print(f'loading weights from {resnest_model_urls[name]}')

  del model.avgpool
  del model.fc

  if weights and weights != 'imagenet':
    print(f'loading weights from {weights}')
    checkpoint = torch.load(weights, map_location="cpu")
    model.load_state_dict(checkpoint['state_dict'], strict=False)

  stages = (
    nn.Sequential(model.conv1, model.bn1, model.relu, model.maxpool),
    model.layer1,
    model.layer2,
    model.layer3,
    model.layer4,
  )

  return model, stages


class Backbone(nn.Module):

  def __init__(
    self,
    model_name,
    weights='imagenet',
    channels=3,
    mode='fix',
    dilated=False,
    strides=None,
    trainable_stem=True,
    trainable_stage4=True,
    trainable_backbone=True,
    backbone_kwargs={},
  ):
    super().__init__()

    self.mode = mode
    self.trainable_stem = trainable_stem
    self.trainable_stage4 = trainable_stage4
    self.trainable_backbone = trainable_backbone
    self.not_training = []
    self.from_scratch_layers = []

    if mode == 'normal':
      self.norm_fn = nn.BatchNorm2d
    elif mode == 'fix':
      self.norm_fn = FixedBatchNorm
    else:
      raise ValueError(f'Unknown mode {mode}. Must be `normal` or `fix`.')

    backbone, stages = build_backbone(
      model_name, dilated, strides, self.norm_fn, weights, channels, **backbone_kwargs,
    )

    self.backbone = backbone
    self.stages = stages

    if not self.trainable_backbone:
      for s in stages:
        set_trainable_layers(s, trainable=False)
      self.not_training.extend(stages)
    else:
      if not self.trainable_stage4:
        self.not_training.extend(stages[:-1])
        for s in stages[:-1]:
          set_trainable_layers(s, trainable=False)

      elif not self.trainable_stem:
        set_trainable_layers(stages[0], trainable=False)
        self.not_training.append(stages[0])

      if self.mode == "fix":
        for s in stages:
          set_trainable_layers(s, torch.nn.BatchNorm2d, trainable=False)
          self.not_training.extend([m for m in s.modules() if isinstance(m, torch.nn.BatchNorm2d)])

  def initialize(self, modules):
    for m in modules:
      if isinstance(m, nn.Conv2d):
        torch.nn.init.kaiming_normal_(m.weight)
      elif isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.constant_(m.bias, 0)
      elif isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm, nn.GroupNorm, nn.LayerNorm)):
        nn.init.constant_(m.weight, 1.0)
        nn.init.constant_(m.bias, 0)

  def get_parameter_groups(self, exclude_partial_names=(), with_names=False):
    names = ([], [], [], [])
    groups = ([], [], [], [])

    scratch_parameters = set()
    all_parameters = set()

    for layer in self.from_scratch_layers:
      for name, param in layer.named_parameters():
        if param in all_parameters:
          continue
        scratch_parameters.add(param)
        all_parameters.add(param)

        if not param.requires_grad:
          continue
        for p in exclude_partial_names:
          if p in name:
            continue

        idx = 2 if "weight" in name else 3
        names[idx].append(name)
        groups[idx].append(param)

    for name, param in self.named_parameters():
      if param in all_parameters:
        continue
      all_parameters.add(param)

      if not param.requires_grad or param in scratch_parameters:
        continue
      for p in exclude_partial_names:
        if p in name:
          continue

      idx = 0 if "weight" in name else 1
      names[idx].append(name)
      groups[idx].append(param)

    if with_names:
      return groups, names

    return groups

  def train(self, mode=True):
    super().train(mode)
    for m in self.not_training:
      m.eval()
    return self


def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM,self).__init__()
        self.p = Parameter(torch.ones(1)*p)
        self.eps = eps

    def forward(self, x):
        return gem(x, p=self.p, eps=self.eps)

    def __repr__(self):
        return self.__class__.__name__ + '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + ', ' + 'eps=' + str(self.eps) + ')'


class CellClassifier(Backbone):

  def __init__(
    self,
    model_name,
    num_classes=20,
    backbone_weights="imagenet",
    channels=3,
    mode='fix',
    dilated=False,
    strides=None,
    trainable_stem=True,
    trainable_stage4=True,
    trainable_backbone=True,
    **backbone_kwargs,
  ):
    super().__init__(
      model_name,
      channels=channels,
      weights=backbone_weights,
      mode=mode,
      dilated=dilated,
      strides=strides,
      trainable_stem=trainable_stem,
      trainable_stage4=trainable_stage4,
      trainable_backbone=trainable_backbone,
      backbone_kwargs=backbone_kwargs,
    )

    self.num_classes = num_classes

    cin = self.backbone.outplanes
    self.classifier = nn.Conv2d(cin, num_classes, 1, bias=False)

    self.from_scratch_layers.extend([self.classifier])
    self.initialize([self.classifier])

    self.pool = GeM()
    self.flatten = nn.Flatten()
    self.dropout = nn.Dropout(p=0.5)

    self.last_linear_cell = nn.Linear(
      in_features=cin, 
      out_features=num_classes)
    self.last_linear_image = nn.Linear(
      in_features=cin, 
      out_features=num_classes)

  def forward(self, x, cnt=16, with_cam=False):
    if with_cam:
      raise NotImplementedError(
        "CAM not currently supported in multi-view mode")

    outs = self.backbone(x)
    features = outs[-1] if isinstance(outs, tuple) else outs

    pooled = self.flatten(self.pool(features))
    viewed_pooled = pooled.view(-1, cnt, pooled.shape[-1])
    viewed_pooled = viewed_pooled.max(1)[0]

    cell_logits = self.last_linear_cell(pooled)
    image_logits = self.last_linear_image(viewed_pooled)

    return cell_logits, image_logits


class CellClassifierV2(Backbone):

  def __init__(
    self,
    model_name,
    num_classes=20,
    backbone_weights="imagenet",
    channels=3,
    mode='fix',
    dilated=False,
    strides=None,
    trainable_stem=True,
    trainable_stage4=True,
    trainable_backbone=True,
    **backbone_kwargs,
  ):
    super().__init__(
      model_name,
      channels=channels,
      weights=backbone_weights,
      mode=mode,
      dilated=dilated,
      strides=strides,
      trainable_stem=trainable_stem,
      trainable_stage4=trainable_stage4,
      trainable_backbone=trainable_backbone,
      backbone_kwargs=backbone_kwargs,
    )

    self.num_classes = num_classes
    cin = self.backbone.outplanes

    self.pool = GeM()
    self.flatten = nn.Flatten()
    self.dropout = nn.Dropout(p=0.5)

    self.last_linear_cell = nn.Linear(
      in_features=cin, 
      out_features=num_classes)
    self.last_linear_image = nn.Linear(
      in_features=cin, 
      out_features=num_classes)
    
    self.from_scratch_layers.extend([self.last_linear_cell, self.last_linear_image])
    self.initialize([self.last_linear_cell, self.last_linear_image])

  def forward(self, x, cnt=16, with_cam=False, cell_logits_to_image_logits=False):
    cnt = torch.tensor(cnt)
    if with_cam:
      raise NotImplementedError(
        "CAM not currently supported in multi-view mode")

    outs = self.backbone(x)
    features = outs[-1] if isinstance(outs, tuple) else outs

    pooled = self.flatten(self.pool(features))

    if cell_logits_to_image_logits:
      cell_logits = self.last_linear_cell(pooled)
      cell_logits_split = torch.split(cell_logits, cnt.tolist())
      image_logits = torch.stack([p.max(0).values for p in cell_logits_split])

      return cell_logits, image_logits

    pooled_split = torch.split(pooled, cnt.tolist())
    pooled_per_img = torch.stack([p.max(0)[0] for p in pooled_split])

    cell_logits = self.last_linear_cell(pooled)
    image_logits = self.last_linear_image(pooled_per_img)

    return cell_logits, image_logits


class ImageClassifier(Backbone):

  def __init__(
    self,
    model_name,
    num_classes=20,
    backbone_weights="imagenet",
    channels=3,
    mode='fix',
    dilated=False,
    strides=None,
    trainable_stem=True,
    trainable_stage4=True,
    trainable_backbone=True,
    **backbone_kwargs,
  ):
    super().__init__(
      model_name,
      channels=channels,
      weights=backbone_weights,
      mode=mode,
      dilated=dilated,
      strides=strides,
      trainable_stem=trainable_stem,
      trainable_stage4=trainable_stage4,
      trainable_backbone=trainable_backbone,
      backbone_kwargs=backbone_kwargs,
    )

    self.num_classes = num_classes

    cin = self.backbone.outplanes
    self.classifier = nn.Conv2d(cin, num_classes, 1, bias=False)

    self.from_scratch_layers.extend([self.classifier])
    self.initialize([self.classifier])

  def forward(self, x, with_cam=False):
    outs = self.backbone(x)
    x = outs[-1] if isinstance(outs, tuple) else outs

    if with_cam:
      features = self.classifier(x)
      logits = gap2d(features)
      return logits, features
    else:
      x = gap2d(x, keepdims=True)
      logits = self.classifier(x).view(-1, self.num_classes)
      return logits

## Load ResNest269 Model

In [12]:
models = []
image_models = []

In [13]:
# WEIGHTS_PATHS = [
#     '/kaggle/input/models/julianamidlej/rs50-f0-c16-citl-pw0-1-e15/pytorch/default/1/hpa2nd-rs50-lr0.0002-b6-aug2nd-adamw-eid-citl-3/model-f0-e14.pth']
# DEVICE = 'cuda'

# for weights_path in WEIGHTS_PATHS:
#     model = CellClassifierV2(
#         'resnest50',
#         19,
#         channels=4,
#         mode='normal',
#         dilated=False,
#         backbone_weights=None,
#     )
#     load_model(model, weights_path, map_location=torch.device(DEVICE))
#     model.eval()

#     models.append(model)

In [14]:
# WEIGHTS_PATHS = [
#     '/kaggle/input/rs269-vanilla-cp/pytorch/default/1/hpa-512-rs269-lr0.1-b32-ls0.1-re-mix-sgd-ema1-cp-1.0-0.5-sum-2.pth']
# DEVICE = 'cuda'

# for weights_path in WEIGHTS_PATHS:
#     model = ImageClassifier(
#         'resnest269',
#         19,
#         channels=4,
#         mode='normal',
#         dilated=False,
#         backbone_weights=None,
#     )
#     load_model(model, weights_path, map_location=torch.device(DEVICE))
#     model.eval()

#     image_models.append(model)

## Loading models
* b3
* b5
* r50d
* r200d
* se50

In [15]:
# model_paths = [
#     '/kaggle/input/b3/pytorch/default/1/b3/checkpoints/f0_epoch-18.pth',
#     '/kaggle/input/b3-f1/pytorch/default/1/b3_F1/checkpoints/f1_epoch-16.pth',
#     '/kaggle/input/b3-f2/pytorch/default/1/b3_F2/checkpoints/f2_epoch-17.pth',
#     '/kaggle/input/b3-f3/pytorch/default/1/b3_F3/checkpoints/f3_epoch-19.pth',
#     '/kaggle/input/b3-f4/pytorch/default/1/b3_F4/checkpoints/f4_epoch-18.pth'
# ]

# for model_path in model_paths:
#     cfg = Config.load_json('/kaggle/input/b3/pytorch/default/1/b3/config.json')
#     model = get_model(cfg).cuda()
#     load_matched_state(model, torch.load(model_path))
#     _ = model.eval()
#     models.append(model)

In [16]:
# model_paths = [
#     # '/kaggle/input/b5/pytorch/default/1/b5/checkpoints/f0_epoch-14.pth',
#     # '/kaggle/input/b5-f1/pytorch/default/1/b5_F1/checkpoints/f1_epoch-14.pth',
#     # '/kaggle/input/b5-f2/pytorch/default/1/b5_F2/checkpoints/f2_epoch-15.pth',
#     # '/kaggle/input/b5-f3-v2/pytorch/default/1/b5_F3/checkpoints/f3_epoch-17.pth',
#     # '/kaggle/input/b5-f4/pytorch/default/1/b5_F4/checkpoints/f4_epoch-14.pth'   
# ]

# for model_path in model_paths:
#     cfg = Config.load_json('/kaggle/input/b5/pytorch/default/1/b5/config.json')
#     model = get_model(cfg).cuda()
#     load_matched_state(model, torch.load(model_path))
#     _ = model.eval()
#     models.append(model)

In [17]:
# model_paths = [
#     # '/kaggle/input/r50-f0/pytorch/default/1/r50_F0/checkpoints/f0_epoch-14.pth',
#     # '/kaggle/input/r50-f1/pytorch/default/1/r50_F1/checkpoints/f1_epoch-14.pth',
#     # '/kaggle/input/r50-f2/pytorch/default/1/r50_F2/checkpoints/f2_epoch-13.pth',
#     # '/kaggle/input/r50-f3/pytorch/default/1/r50_F3/checkpoints/f3_epoch-15.pth',
#     # '/kaggle/input/r50-f4/pytorch/default/1/r50_F4/checkpoints/f4_epoch-14.pth'
# ]

# for model_path in model_paths:
#     cfg = Config.load_json('/kaggle/input/r50-f0/pytorch/default/1/r50_F0/config.json')
#     model = get_model(cfg).cuda()
#     load_matched_state(model, torch.load(model_path))
#     _ = model.eval()
#     models.append(model)

In [18]:
!cp ../input/landmark-additional-packages/resnet200d_ra2-bdba9bf9.pth /root/.cache/torch/hub/checkpoints/

In [19]:
model_paths = [
    # '/kaggle/input/r200/pytorch/default/1/r200d/checkpoints/f0_epoch-13.pth',
    # '/kaggle/input/r200-f1/pytorch/default/1/r200d_F1/checkpoints/f1_epoch-15.pth',
    # '/kaggle/input/r200-f2/pytorch/default/1/r200d_F2/checkpoints/f2_epoch-15.pth',
    # '/kaggle/input/r200-f3/pytorch/default/1/r200d_F3/checkpoints/f3_epoch-14.pth',
    '/kaggle/input/r200-f4/pytorch/default/1/r200d_F4/checkpoints/f4_epoch-14.pth'
]

for model_path in model_paths:
    cfg = Config.load_json('/kaggle/input/r200/pytorch/default/1/r200d/config.json')
    model = get_model(cfg).cuda()
    load_matched_state(model, torch.load(model_path))
    _ = model.eval()
    models.append(model)

[ ! ] Init a RANZCRResNet200D, mdl_name: resnet200d, pool: AdaptiveAvgPool2d
[ √ ] All layers are loaded


In [20]:
# model_paths = [
#     # '/kaggle/input/se50/pytorch/default/1/se50/checkpoints/f0_epoch-18.pth',
#     # '/kaggle/input/se50-f1/pytorch/default/1/se50_F1/checkpoints/f1_epoch-19.pth',
#     # '/kaggle/input/se50-f2/pytorch/default/1/se50_F2/checkpoints/f2_epoch-18.pth',
#     # '/kaggle/input/se50-f3/pytorch/default/1/se50_F3/checkpoints/f3_epoch-18.pth',
#     # '/kaggle/input/se50-f4/pytorch/default/1/se50_F4/checkpoints/f4_epoch-18.pth'
# ]

# for model_path in model_paths:
#     cfg = Config.load_json('/kaggle/input/se50/pytorch/default/1/se50/config.json')
#     model = get_model(cfg).cuda()
#     load_matched_state(model, torch.load(model_path))
#     _ = model.eval()
#     models.append(model)

In [21]:
print(len(models))
print(len(image_models))

1
0


## Cell Segmentator

In [22]:
# ! pip install -q "/kaggle/input/pycocotools/wheels/pycocotools-2.0.6-cp310-cp310-linux_x86_64.whl"
# ! cp -r /kaggle/input/hpapytorchzoozip/pytorch_zoo-master .
# ! pip install -q "./pytorch_zoo-master"
# ! cp -r "/kaggle/input/hpacellsegmentatormaster/HPA-Cell-Segmentation-master" .
# ! pip install -q "./HPA-Cell-Segmentation-master"

In [23]:
# # Cell Segmentator Tool
# print("\n... INSTALLING AND IMPORTING CELL-PROFILER TOOL (HPACELLSEG) ...\n")

# try:
#     import hpacellseg.cellsegmentator as cellsegmentator
#     from hpacellseg.utils import label_cell
# except:
#     ! pip install -q "/kaggle/input/pycocotools/wheels/pycocotools-2.0.6-cp310-cp310-linux_x86_64.whl"
#     ! cp -r /kaggle/input/hpapytorchzoozip/pytorch_zoo-master .
#     ! pip install -q "./pytorch_zoo-master"
#     ! cp -r "/kaggle/input/hpacellsegmentatormaster/HPA-Cell-Segmentation-master" .
#     ! pip install -q "./HPA-Cell-Segmentation-master"

#     import hpacellseg.cellsegmentator as cellsegmentator
#     from hpacellseg.utils import label_cell

# # Install dependencies
# # ! pip -qq install 'scikit-image<0.15'
# # ! pip -qq install pycocotools
# # ! pip -qq install git+https://github.com/CellProfiling/HPA-Cell-Segmentation

# # Imports
# import os
# import copy
# import pickle
# from math import ceil
# from itertools import groupby
# import base64
# import typing as t
# import zlib
# import random

# import cv2
# import torch
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# from tqdm import tqdm
# from pycocotools import mask as mutils
# from pycocotools import _mask as coco_mask

# cv2.setNumThreads(0)

In [24]:
# # Cell Segmentator

# NUC_MODEL = '../input/hpacellsegmentatormodelweights/dpn_unet_nuclei_v1.pth'
# CELL_MODEL = '../input/hpacellsegmentatormodelweights/dpn_unet_cell_3ch_v1.pth'

# import collections
# collections.Iterable = collections.abc.Iterable
# np.bool = np.bool_


# import torch
# import torch.nn
# import torch.nn.functional as F
# import hpacellseg.cellsegmentator as cellsegmentator
# from skimage import transform as sk_transform, util as sk_util

# NORMALIZE = {"mean": [124 / 255, 117 / 255, 104 / 255], "std": [1 / (0.0167 * 255)] * 3}

# class CellSegmentator(cellsegmentator.CellSegmentator):
#     def pred_nuclei(self, images):
#         def _preprocess(image):
#             if isinstance(image, str):
#                 image = imageio.imread(image)
#             self.target_shape = image.shape
#             if len(image.shape) == 2:
#                 image = np.dstack((image, image, image))
#             image = sk_transform.rescale(image, self.scale_factor, multichannel=True)
#             nuc_image = np.dstack((image[..., 2], image[..., 2], image[..., 2]))
#             if self.padding:
#                 rows, cols = nuc_image.shape[:2]
#                 self.scaled_shape = rows, cols
#                 nuc_image = cv2.copyMakeBorder(
#                     nuc_image,
#                     32,
#                     (32 - rows % 32),
#                     32,
#                     (32 - cols % 32),
#                     cv2.BORDER_REFLECT,
#                 )
#             nuc_image = nuc_image.transpose([2, 0, 1])
#             return nuc_image

#         def _segment_helper(imgs):
#             with torch.no_grad():
#                 mean = torch.as_tensor(NORMALIZE["mean"], device=self.device)
#                 std = torch.as_tensor(NORMALIZE["std"], device=self.device)
#                 imgs = torch.tensor(np.array(imgs)).float()
#                 imgs = imgs.to(self.device)
#                 imgs = imgs.sub_(mean[:, None, None]).div_(std[:, None, None])

#                 imgs = self.nuclei_model(imgs)
#                 imgs = F.softmax(imgs, dim=1)
#                 return imgs

#         preprocessed_imgs = map(_preprocess, images)
#         predictions = map(lambda x: _segment_helper([x]), preprocessed_imgs)
#         predictions = map(lambda x: x.to("cpu").numpy()[0], predictions)
#         predictions = map(sk_util.img_as_ubyte, predictions)
#         predictions = list(map(self._restore_scaling_padding, predictions))
#         return predictions

#     def pred_cells(self, images, precombined=False):
#         def _preprocess(image):
#             self.target_shape = image.shape
#             if not len(image.shape) == 3:
#                 raise ValueError("image should has 3 channels")
#             cell_image = sk_transform.rescale(image, self.scale_factor, multichannel=True)
#             if self.padding:
#                 rows, cols = cell_image.shape[:2]
#                 self.scaled_shape = rows, cols
#                 cell_image = cv2.copyMakeBorder(
#                     cell_image,
#                     32,
#                     (32 - rows % 32),
#                     32,
#                     (32 - cols % 32),
#                     cv2.BORDER_REFLECT,
#                 )
#             cell_image = cell_image.transpose([2, 0, 1])
#             return cell_image

#         def _segment_helper(imgs):
#             with torch.no_grad():
#                 mean = torch.as_tensor(NORMALIZE["mean"], device=self.device)
#                 std = torch.as_tensor(NORMALIZE["std"], device=self.device)
#                 imgs = torch.tensor(np.array(imgs)).float()
#                 imgs = imgs.to(self.device)
#                 imgs = imgs.sub_(mean[:, None, None]).div_(std[:, None, None])

#                 imgs = self.cell_model(imgs)
#                 imgs = F.softmax(imgs, dim=1)
#                 return imgs

#         if not precombined:
#             images = self._image_conversion(images)
#         preprocessed_imgs = map(_preprocess, images)
#         predictions = map(lambda x: _segment_helper([x]), preprocessed_imgs)
#         predictions = map(lambda x: x.to("cpu").numpy()[0], predictions)
#         predictions = map(self._restore_scaling_padding, predictions)
#         predictions = list(map(sk_util.img_as_ubyte, predictions))

#         return predictions

In [25]:
# segmentator = CellSegmentator(
#     NUC_MODEL,
#     CELL_MODEL,
#     scale_factor=0.25,
#     device=DEVICE,
#     padding=True,
#     multi_channel_model=True,
# )

In [26]:
# import os
# import glob
# from pathlib import Path

# # Define your root and test image directory
# ROOT = '../input/hpa-single-cell-image-classification'
# TEST_IMAGES_DIR = f"{ROOT}/test/"

# print(f"Listing images in: {TEST_IMAGES_DIR}")

# # This method lists all files and directories, so you need to filter for files
# images_set = set()
# if os.path.exists(TEST_IMAGES_DIR):
#     for filename in os.listdir(TEST_IMAGES_DIR):
#         if filename.endswith(".png") or filename.endswith(".jpg"): # Assuming image formats
#             images_set.add(filename.split('_')[0])
#     print(f"\n(os.listdir): Found {len(images_set)} images.")
# else:
#     print(f"Directory not found: {TEST_IMAGES_DIR}")

In [27]:
# def load_all_images_dict(images_dir, sample_ids):
#     image_dict = {}

#     for sample_id in tqdm(sample_ids, desc="Loading images"):
#         try:
#             image = get_image(images_dir, sample_id)
#             image_dict[sample_id] = image
#         except Exception as e:
#             print(f"Erro ao carregar {sample_id}: {e}")
#             continue

#     return image_dict

In [28]:
# image_dict = load_all_images_dict(TEST_IMAGES_DIR, images_set)

In [29]:
# def get_masks_from_images(image_ids, imgs):
#     cell_masks = []
    
#     # Iterar por cada imagem individualmente e monitorar o progresso com tqdm
#     for image_id, img in tqdm(zip(image_ids, imgs), desc="Generating cell masks", total=len(image_ids)):
#         # Separando as imagens em diferentes canais (R, G, B)
#         image = [
#             img[:, :, 0],  # canal vermelho
#             img[:, :, 3],  # canal amarelo (presumido)
#             img[:, :, 2],  # canal azul
#         ]

#         # Segmentação de núcleos
#         nuc_segmentations = segmentator.pred_nuclei([image[2]])
#         # Segmentação de células
#         cell_segmentations = segmentator.pred_cells(image)

#         # Gerando a máscara da célula
#         _, cell_mask = label_cell(nuc_segmentations, cell_segmentations)
#         cell_masks.append(cell_mask)

#     return cell_masks


In [30]:
# image_ids = list(image_dict.keys())
# images = list(image_dict.values())

# cell_masks = get_masks_from_images(image_ids, images)

## If we read from a csv

In [31]:
df = pd.read_csv('/kaggle/input/hpa-best-segmentation-take-use/best_segmentation_result.csv')
# df = pd.read_csv('/kaggle/input/segmentation-masks-lucas-baseline/submission.csv')

In [32]:
df

,ID,ImageWidth,ImageHeight,PredictionString
0,0040581b-f1f2-4fbe-b043-b6bfea5404bb,2048,2048,0 0.38977348804473877 eNq9Uz0LQjEM/EspdOjQMUOH...
1,004a270d-34a2-4d60-bbe4-365fca868193,2048,2048,0 0.0026059572119265795 eNqNltF20zAMhl9JgQxymD...
2,00537262-883c-4b37-a3a1-a4931b6faea5,2048,2048,0 0.0024114057887345552 eNrLCUwwijHKMcjxS8kyDD...
3,00c9a1c9-2f06-476f-8b0d-6d01032874a2,2048,2048,0 0.00609578937292099 eNrLCA0IM7RJyTc0AAEfCx+s...
4,0173029a-161d-40ef-af28-2342915b22fb,3072,3072,0 0.01869528740644455 eNrFU00PgjAM/Us4Ju7AkUSj...
...,...,...,...,...
554,fea47298-266a-4cf4-93bd-55d1bcc2fc7d,1728,1728,0 0.004143985453993082 eNqVVE1vwyAM/Uv2ZKkcIpV...
555,feb955db-6c07-4717-a98b-92236c8e01d8,2048,2048,0 0.004161765333265066 eNqlVF8LgyAQ/0q63YOMBjF...
556,fefb9bb7-934a-40d1-8d2f-210265857388,2048,2048,0 0.11587181687355042 eNozCLHOSTE0AAMTHwOygI8J...
557,ff069fa2-d948-408e-91b3-034cfea428d1,3072,3072,0 0.4128116965293884 eNq9VD0PgyAQ/UsXegODowMDj...


In [33]:
# Lista para armazenar os dicionários com informações de cada célula
imgs = []

# Itera sobre todas as linhas do DataFrame df
for i, x in df.iterrows():
    
    # Extrai os labels da string PredictionString (um a cada 3 itens)
    label = x.PredictionString.split(' ')[0::3]
    
    # Extrai as probabilidades associadas aos labels (um a cada 3 itens)
    prob = x.PredictionString.split(' ')[1::3]
    
    # Extrai as máscaras codificadas em RLE (um a cada 3 itens)
    encodes = x.PredictionString.split(' ')[2::3]
    
    # Itera sobre os RLEs únicos (remove duplicatas com set)
    for idx, enc in enumerate(list(set(encodes))):
        imgs.append({
            'image_id': x.ID,            # ID da imagem original
            'cell_id': idx + 1,          # Índice da célula (começa do 1)
            'enc': enc,                  # Máscara codificada em RLE
            'fname': f'{x.ID}_{idx+1}',  # Nome do arquivo no formato <image_id>_<cell_id>
        })

# Converte a lista de dicionários em um novo DataFrame
tm = pd.DataFrame(imgs)

In [34]:
tm

,image_id,cell_id,enc,fname
0,0040581b-f1f2-4fbe-b043-b6bfea5404bb,1,eNqNlN1u4jAQRl/JQ0zLBSuttKlESbDdMsBAvJIlDMyybt...,0040581b-f1f2-4fbe-b043-b6bfea5404bb_1
1,0040581b-f1f2-4fbe-b043-b6bfea5404bb,2,eNq9Uz0LQjEM/EspdOjQMUOHh4oILm/o8EBB9P+PorGFXA...,0040581b-f1f2-4fbe-b043-b6bfea5404bb_2
2,0040581b-f1f2-4fbe-b043-b6bfea5404bb,3,eNrLSUjIMkoxjMkzNAADEx8w5ZNg4GKADHxMDPDyCYkTBF...,0040581b-f1f2-4fbe-b043-b6bfea5404bb_3
3,0040581b-f1f2-4fbe-b043-b6bfea5404bb,4,eNqdUk1rxCAQ/UtOMDSHPewhh7RYlTIHCxInRahQy/7/W8...,0040581b-f1f2-4fbe-b043-b6bfea5404bb_4
4,0040581b-f1f2-4fbe-b043-b6bfea5404bb,5,eNqdlNGKKjEMhl9pMg4yC93Dsg4o2KndQy+6WJwI4VgOxf...,0040581b-f1f2-4fbe-b043-b6bfea5404bb_5
...,...,...,...,...
9335,ff23eea9-4bbe-42af-a8da-9ae16321fc6d,20,eNqNU8tuwyAQ/CW2YMSBow9OiogP1MItlWi9kUhF///aOG...,ff23eea9-4bbe-42af-a8da-9ae16321fc6d_20
9336,ff23eea9-4bbe-42af-a8da-9ae16321fc6d,21,eNqNk8FOwzAMhl8p3gIrYgcOO5SqsgLKIdq8kW5GyyHA+9...,ff23eea9-4bbe-42af-a8da-9ae16321fc6d_21
9337,ff23eea9-4bbe-42af-a8da-9ae16321fc6d,22,eNqNUttKxDAQ/aUZNmBh+7BIwIAxjRAhYqzpGnC0Yf3/N6...,ff23eea9-4bbe-42af-a8da-9ae16321fc6d_22
9338,ff23eea9-4bbe-42af-a8da-9ae16321fc6d,23,eNqNUkFuwyAQ/JIpHKjtSj04FQdKqEQTDlQQiUpEQsr/b7...,ff23eea9-4bbe-42af-a8da-9ae16321fc6d_23


In [35]:
df = df.drop(columns='PredictionString')
df.head()

,ID,ImageWidth,ImageHeight
0,0040581b-f1f2-4fbe-b043-b6bfea5404bb,2048,2048
1,004a270d-34a2-4d60-bbe4-365fca868193,2048,2048
2,00537262-883c-4b37-a3a1-a4931b6faea5,2048,2048
3,00c9a1c9-2f06-476f-8b0d-6d01032874a2,2048,2048
4,0173029a-161d-40ef-af28-2342915b22fb,3072,3072


In [36]:
sample_submission = pd.read_csv('../input/hpa-single-cell-image-classification/sample_submission.csv', index_col=0)
sample_submission.head()

,ImageWidth,ImageHeight,PredictionString
ID,,,
0040581b-f1f2-4fbe-b043-b6bfea5404bb,2048,2048,0 1 eNoLCAgIMAEABJkBdQ==
004a270d-34a2-4d60-bbe4-365fca868193,2048,2048,0 1 eNoLCAgIMAEABJkBdQ==
00537262-883c-4b37-a3a1-a4931b6faea5,2048,2048,0 1 eNoLCAgIMAEABJkBdQ==
00c9a1c9-2f06-476f-8b0d-6d01032874a2,2048,2048,0 1 eNoLCAgIMAEABJkBdQ==
0173029a-161d-40ef-af28-2342915b22fb,3072,3072,0 1 eNoLCAgIsAQABJ4Beg==


In [37]:
class SliceInferenceDataset(torch.utils.data.Dataset):
    def __init__(self, df, tta=16, cfg=None, tfms=None):
        self.df = df                                  # DataFrame contendo informações de imagem_id e codificação da máscara
        self.iids = self.df.image_id.unique()         # Lista única de IDs de imagem
        self.tta = tta                                # Número de Test Time Augmentations a serem aplicados
        
    def __len__(self):
        return len(self.iids)                         # Número total de imagens únicas

    def __getitem__(self, idx):
        iid = self.iids[idx]                          # Pega o image_id pelo índice
        
        # Define uma seed fixa para garantir reproducibilidade por amostra
        seed = 42
        random.seed(seed)
        np.random.seed(seed)
        
        # Lê os canais de imagem correspondentes a cada cor (em escala de cinza)
        mt = f'../input/hpa-single-cell-image-classification/test/{iid}_red.png'
        er = f'../input/hpa-single-cell-image-classification/test/{iid}_yellow.png'
        nu = f'../input/hpa-single-cell-image-classification/test/{iid}_blue.png'
        pr = f'../input/hpa-single-cell-image-classification/test/{iid}_green.png'
        r = cv2.imread(mt, 0).astype(np.float) / 255.0
        g = cv2.imread(pr, 0).astype(np.float) / 255.0
        b = cv2.imread(nu, 0).astype(np.float) / 255.0
        a = cv2.imread(er, 0).astype(np.float) / 255.0
        
        sz = r.shape[0]                               # Assume imagem quadrada (W = H)
        img = np.stack([r, g, b, a], -1)              # Empilha os 4 canais para criar imagem 4-channel RGBY
        
        sli = []                                       # Lista para armazenar os crops das células (imagem + fname)
        
        # Itera sobre todas as células daquela imagem
        for i, x in self.df[self.df.image_id == iid].iterrows():
            # Decodifica a máscara RLE
            bd = base64.b64decode(x.enc)
            zd = zlib.decompress(bd)
            encoded = [{'counts': zd, 'size': (sz, sz)}]
            ded = coco_mask.decode(encoded)[:, :, 0]  # Máscara binária

            # Se não tiver célula detectada
            if len(np.unique(ded)) == 1:
                continue

            # Extrai bounding box da máscara
            xr, yr = np.where(ded == 1)
            sub = img[xr.min(): xr.max(), yr.min(): yr.max()]  # Recorta imagem
            crop_sub_mask = ded[xr.min(): xr.max(), yr.min(): yr.max()]
            crop_sub_mask = np.repeat(crop_sub_mask[:, :, np.newaxis], 4, axis=2)  # Expande máscara para 4 canais
            
            # Aplica máscara à imagem
            r = sub * crop_sub_mask

            # Ajusta para quadrado e redimensiona para 256×256
            sli.append((cv2.resize(squarify(r, 0), (256, 256)).astype(np.float32), x.fname))

        # Se nenhuma célula válida foi extraída da imagem, retorna None
        if not sli:
            return None
        
        BS, tta = len(sli) + 1, self.tta  # BS: batch size estimado (número de células + 1)
        ipts = []                         # Lista de batches de células com TTA

        raw_ipt = [e[0] for e in sli]     # Apenas as imagens (sem o fname)

        # Aplica transformações de TTA
        if tta == 1:
            ipts.append(torch.stack([tensor_tfms(res_tfms(image=x)['image']) for x in raw_ipt]).float())
        else:
            for tt in range(tta):
                ipts.append(torch.stack([tensor_tfms(tta_tfms(image=x)['image']) for x in raw_ipt]).float())

        # Também transforma a imagem inteira para outro uso (ex: CAM)
        image = image_tfms(image_res_tfms(image=img)['image'])
            
        # Retorna:
        # - imagem da amostra inteira (RGBY)
        # - células da imagem após TTA
        # - batch size estimado
        # - número de células detectadas
        # - número de TTA
        # - image_id
        # - lista dos nomes de arquivos para cada célula
        return image, ipts, BS, len(sli), tta, iid, [x[1] for x in sli]


In [38]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

sid = SliceInferenceDataset(tm, tta=8)
dl = torch.utils.data.DataLoader(sid, batch_size=1, num_workers=2, worker_init_fn=seed_worker)

In [39]:
# Move os modelos para GPU e define modo de avaliação (desativa dropout, batchnorm)
models = [model.cuda().eval() for model in models]

# Listas para armazenar predições
pdfs = []        # Para armazenar predições por célula
whole_dfs = []   # Para armazenar predições por imagem (whole image prediction)

# Loop sobre o DataLoader de inferência (dl)
for image, ipts, BS, lsli, tta, iid, fnames_raw in tqdm.tqdm(dl):
    # Extração e formatação das variáveis
    BS = BS.item()           # Tamanho do batch
    # print(BS)
    tta = tta.item()         # Número de Test Time Augmentations
    iid = iid[0]             # image_id (string)
    fnames = [e[0] for e in fnames_raw]  # Nomes dos arquivos das células
    lsli = lsli.item()       # Número de células na imagem
    # print(lsli)
    
    predicted_ps = []  # Lista para armazenar as predições de células
    exp_ps = []        # Lista para armazenar as predições da imagem inteira

    # Processamento das células em blocos do tamanho do batch
    for i in range(0, lsli, BS):
        with torch.no_grad():
            res = []  # Predições por célula (para cada modelo com TTA)
            exp = []  # Predições da imagem inteira (para cada modelo)
            
            # Inferência da imagem completa com modelos de imagem (ex: Puzzle-CAM)
            for model in image_models:
                image = image.float()  # Garante que o tipo é float32
                model = model.float()  # Idem para o modelo
                exp.append(model(image))  # Predição da imagem

            # Inferência por TTA: aplica vários modelos às células com augmentations
            for tt in range(tta):
                ipt = ipts[tt][0].cuda()  # Recupera as células para essa TTA e move para GPU
                for model in models:
                    with torch.cuda.amp.autocast():  # Inferência com mixed-precision
                        ifr = model(ipt, len(ipt))   # Modelo retorna predição por célula + imagem
                    res.append(ifr[0].float())  # Coleta saída da célula
                    exp.append(ifr[1].float())  # Coleta saída da imagem

        # Média das predições de cada célula com sigmoid (multi-label)
        predict_p = [torch.sigmoid(r.cpu()) for r in res]
        exp_p = [torch.sigmoid(r.cpu()) for r in exp]
        predict_p = np.stack(predict_p).mean(0)  # Média ao longo de modelos/TTA
        exp_p = np.stack(exp_p).mean(0)          # Média ao longo de modelos/TTA

        predicted_ps.append(predict_p)  # Armazena predições das células
        exp_ps.append(exp_p)            # Armazena predições da imagem

    # Concatena predições por célula (final do loop da imagem)
    p = np.concatenate(predicted_ps)  # Shape: (n_cells, n_classes)
    image_df = pd.DataFrame(p, index=fnames)  # DataFrame com scores para cada célula
    whole_df = pd.DataFrame(
        np.concatenate(exp_ps).mean(0).reshape(1, 19), index=[iid]
    )  # DataFrame com score médio da imagem inteira

    # Salva resultados
    whole_dfs.append(whole_df)  # Uma linha por imagem
    pdfs.append(image_df)       # Uma linha por célula

100%|██████████| 559/559 [19:19<00:00,  2.07s/it]


In [40]:
# Concatena todos os DataFrames de predição por imagem (gerados durante a inferência)
# Isso resulta em um DataFrame onde cada linha representa uma imagem (image_id) com predições para cada classe
image_level = pd.concat(whole_dfs)

# Junta o DataFrame `image_level` com o DataFrame `tm`, para associar os image_id com seus nomes de arquivo (fname)
# - `reset_index()` move o índice (que contém image_id) para uma coluna chamada 'index'
# - `merge(..., how='left')` garante que todos os dados de `image_level` sejam mantidos
# - `on='index'` (de `image_level`) se conecta com `image_id` (de `tm`)
image_pred = image_level.reset_index().merge(
    tm[['image_id', 'fname']],  # Subconjunto com apenas as colunas relevantes do `tm`
    left_on='index',            # Usa o antigo índice como chave de junção
    right_on='image_id',        # Faz a junção com a coluna image_id do `tm`
    how='left'                  # Preserva todas as linhas de `image_level`
)
# Organiza o DataFrame final:
# - Define 'fname' como índice, o que facilita o acesso por nome de célula depois
# - Remove as colunas 'index' (image_id original do image_level) e 'image_id' (duplicado do tm)
image_pred = image_pred.set_index('fname').drop(['index', 'image_id'], axis=1)

In [41]:
# Concatena todos os DataFrames de predições por célula (armazenados em `pdfs`)
# Cada DataFrame em `pdfs` contém predições de classes para as células daquela imagem, indexadas por fname (ex: "abc_1")
pub_pred = pd.concat(pdfs)

# Multiplica as predições por célula (`pub_pred`) pelas predições a nível de imagem (`image_pred`)
# - Ambas têm o mesmo índice (fname, como "abc_1"), então a operação é feita elemento a elemento
# - Isso pode ser interpretado como um reponderamento das predições da célula com base na imagem
#   (por exemplo, se a imagem não tiver forte ativação para uma classe, a predição da célula será atenuada)
merge_pred = pub_pred * image_pred.loc[pub_pred.index]

## If any ensemble

In [42]:
merge_pred.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0040581b-f1f2-4fbe-b043-b6bfea5404bb_1,0.011618,7.301239e-07,0.007113,0.006608,3.600785e-06,0.000070,0.000045,0.000047,0.000012,0.000007,1.864443e-06,2.406357e-07,0.000377,0.000125,0.000008,6.663824e-06,0.023899,0.030088,4.146776e-05
0040581b-f1f2-4fbe-b043-b6bfea5404bb_2,0.026791,8.251444e-07,0.002801,0.002326,5.362818e-06,0.000081,0.000012,0.000040,0.000007,0.000011,2.195284e-06,1.603905e-07,0.000336,0.000096,0.000003,4.581566e-06,0.019350,0.116015,1.569189e-05
0040581b-f1f2-4fbe-b043-b6bfea5404bb_3,0.078992,7.497535e-08,0.001400,0.000599,6.473358e-07,0.000025,0.000002,0.000068,0.000003,0.000005,2.324250e-07,2.260056e-08,0.000650,0.000048,0.000003,3.851995e-06,0.008081,0.215715,3.106622e-06
0040581b-f1f2-4fbe-b043-b6bfea5404bb_4,0.071834,1.534099e-07,0.000997,0.000735,7.697016e-07,0.000015,0.000001,0.000036,0.000002,0.000002,7.395872e-07,2.298854e-08,0.000085,0.000045,0.000001,8.728995e-07,0.011910,0.442827,4.257956e-08
0040581b-f1f2-4fbe-b043-b6bfea5404bb_5,0.008910,3.482999e-07,0.000762,0.001253,2.146697e-06,0.000195,0.000011,0.000016,0.000007,0.000012,5.251300e-07,6.715247e-08,0.000574,0.000080,0.000002,5.445857e-06,0.013925,0.023693,4.617271e-05


## Save prediction

In [43]:
df.head()

,ID,ImageWidth,ImageHeight
0,0040581b-f1f2-4fbe-b043-b6bfea5404bb,2048,2048
1,004a270d-34a2-4d60-bbe4-365fca868193,2048,2048
2,00537262-883c-4b37-a3a1-a4931b6faea5,2048,2048
3,00c9a1c9-2f06-476f-8b0d-6d01032874a2,2048,2048
4,0173029a-161d-40ef-af28-2342915b22fb,3072,3072


In [44]:
df = df.set_index('ID') # Define a coluna 'ID' como índice do DataFrame `df`
merge_pred.index.name = 'fname' # Define o nome do índice de `merge_pred` como 'fname'
merge_pred = merge_pred.reset_index() # Reseta o índice de `merge_pred`, transformando 'fname' em uma coluna normal
tm = tm.set_index('fname') # Define a coluna 'fname' como índice do DataFrame `tm`

# Cria uma nova coluna 'ID' em `merge_pred`, extraindo a parte antes do "_" da coluna 'fname'
# Exemplo: se fname = "abc_3", então ID = "abc"
merge_pred['ID'] = merge_pred['fname'].str.split('_', expand=True)[0]

In [45]:
# Lista para armazenar as predições finais formatadas por imagem
j_pred = []

# Itera sobre todos os IDs únicos (ou seja, imagens únicas) presentes no DataFrame `merge_pred`
for iid in merge_pred.ID.unique():
    enc = ''  # Inicializa a string de predição para essa imagem

    # Filtra `merge_pred` para obter apenas as linhas (células) associadas ao ID atual (imagem atual)
    sub_df = merge_pred[merge_pred.ID == iid]

    # Para cada célula da imagem
    for idx, row in sub_df.iterrows():
        # Para cada uma das 19 classes (0 a 18), cria um trecho do tipo: "class prob mask"
        for i in range(19):
            enc += f'{i} {row[i]} {tm.loc[row.fname].enc} '
    
    # Adiciona a predição formatada como um dicionário na lista `j_pred`
    # Remove o último espaço de `enc` com `[:-1]`
    j_pred.append({
        'ID': iid,  # ID da imagem
        'ImageWidth': df.loc[iid].ImageWidth,   # Largura da imagem (obtida do DataFrame original `df`)
        'ImageHeight': df.loc[iid].ImageHeight, # Altura da imagem
        'PredictionString': enc[:-1]  # String com todas as predições no formato esperado para submissão
    })


In [46]:
# Converte a lista de dicionários `j_pred` em um DataFrame do pandas
fast_sub = pd.DataFrame(j_pred)

# Salva esse DataFrame em um arquivo CSV chamado 'pub.csv'
# Esse arquivo será formatado como exigido para submissão na competição (ID, ImageWidth, ImageHeight, PredictionString)
fast_sub.to_csv('pub.csv', index=False)

# Define a coluna 'ID' como índice do DataFrame (útil para buscas e manipulações futuras)
fast_sub = fast_sub.set_index('ID')


In [47]:
fast_sub.head(2)

,ImageWidth,ImageHeight,PredictionString
ID,,,
0040581b-f1f2-4fbe-b043-b6bfea5404bb,2048,2048,0 0.011617940850555897 eNqNlN1u4jAQRl/JQ0zLBSu...
004a270d-34a2-4d60-bbe4-365fca868193,2048,2048,0 7.722818736510817e-06 eNqdlMFqwzAMhl9JgkBN8a...


## save

In [48]:
# Substitui as previsões do DataFrame `sample_submission` pelas do `fast_sub`
# drop(fast_sub.index): remove as linhas de 'sample_submission' que já estão em 'fast_sub' (com mesmo ID)
# depois concatena com 'fast_sub' para manter as novas previsões
sub2 = pd.concat([sample_submission.drop(fast_sub.index), fast_sub], axis=0)

# Reordena as linhas para seguir exatamente a ordem original de 'sample_submission'
sub2 = sub2.loc[sample_submission.index]

# Salva o novo DataFrame como CSV no formato de submissão exigido
sub2.to_csv('submission.csv')